# STAT 764 · Capstone Project 

Jake

In [17]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:                       # you are inside (or just outside) your clone
    sys.path.insert(0, str(found[0] / "course"))
else:                           # Colab, or a copy saved outside the clone
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

from stat764 import load

In [18]:
import numpy as np
import pandas as pd

# Load the TidyTuesday hotels dataset directly into a Pandas DataFrame
hotels = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-02-11/hotels.csv')

# Verify it loaded correctly
print(hotels.head())
print("Shape:", hotels.shape)

          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                        27                          1   
4                        27                          1   

   stays_in_weekend_nights  stays_in_week_nights  adults  ...  deposit_type  \
0                        0                     0       2  ...    No Deposit   
1     

In [19]:
# Cancellation rate across all rows (dataset column is spelled is_canceled).
canceled = hotels["is_canceled"].eq(1)
print(f"Cancellation rate: {canceled.mean():.2%} ({canceled.sum():,}/{len(hotels):,} rows)")

# Count missing values for every variable, including columns with zero missing values.
print("\nMissing values by variable:")
print(hotels.isna().sum().to_string())

Cancellation rate: 37.04% (44,224/119,390 rows)

Missing values by variable:
hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type         

In [20]:
# Libraries
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split

In [21]:
# Count exact row patterns, from most to least frequent
row_counts = (
    hotels.groupby(hotels.columns.tolist(), dropna=False, sort=False)
    .size()
    .sort_values(ascending=False)
)

print("Most repeated exact rows:")
print(row_counts.head(10).to_string())
print(f"\nRows beyond the first copy: {hotels.duplicated().sum():,}")

Most repeated exact rows:
hotel       is_canceled  lead_time  arrival_date_year  arrival_date_month  arrival_date_week_number  arrival_date_day_of_month  stays_in_weekend_nights  stays_in_week_nights  adults  children  babies  meal  country  market_segment  distribution_channel  is_repeated_guest  previous_cancellations  previous_bookings_not_canceled  reserved_room_type  assigned_room_type  booking_changes  deposit_type  agent  company  days_in_waiting_list  customer_type  adr    required_car_parking_spaces  total_of_special_requests  reservation_status  reservation_status_date
City Hotel  1            277        2016               November            46                        7                          1                        2                     2       0.0       0       BB    PRT      Groups          TA/TO                 0                  0                       0                               A                   A                   0                Non Refund    NaN    NaN    

# Removing Duplicated Rows

In [22]:
print(f"Rows belonging to duplicate groups: {hotels.duplicated(keep=False).sum():,}")
before = len(hotels)
hotels = hotels.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(hotels):,} duplicate rows; {len(hotels):,} rows remain.")

Rows belonging to duplicate groups: 40,165
Removed 31,994 duplicate rows; 87,396 rows remain.


# Deposit Leaks

In [ ]:
# Remove deposit_type because it can leak information about cancellation.
hotels = hotels.drop(columns=["deposit_type"])

In [24]:
# Cancellation rate across all rows (dataset column is spelled is_canceled).
canceled = hotels["is_canceled"].eq(1)
print(f"Cancellation rate: {canceled.mean():.2%} ({canceled.sum():,}/{len(hotels):,} rows)")

# Count missing values for every variable, including columns with zero missing values.
print("\nMissing values by variable:")
print(hotels.isna().sum().to_string())

Cancellation rate: 27.49% (24,025/87,396 rows)

Missing values by variable:
hotel                                 0
is_canceled                           0
lead_time                             0
arrival_date_year                     0
arrival_date_month                    0
arrival_date_week_number              0
arrival_date_day_of_month             0
stays_in_weekend_nights               0
stays_in_week_nights                  0
adults                                0
children                              4
babies                                0
meal                                  0
country                             452
market_segment                        0
distribution_channel                  0
is_repeated_guest                     0
previous_cancellations                0
previous_bookings_not_canceled        0
reserved_room_type                    0
assigned_room_type                    0
booking_changes                       0
agent                             12193
comp

In [25]:
arrival_date = pd.to_datetime(
    hotels["arrival_date_year"].astype(str)
    + "-"
    + hotels["arrival_date_month"]
    + "-"
    + hotels["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d",
)

edges = pd.date_range(arrival_date.min(), arrival_date.max(), periods=11)
fold_id = pd.cut(arrival_date, bins=edges, labels=False, include_lowest=True)

# Ten DataFrames, ordered from earliest to latest arrival period
time_folds = [
    hotels.loc[fold_id.eq(i)].copy()
    for i in range(10)
]

for i, fold in enumerate(time_folds, start=1):
    dates = arrival_date.loc[fold.index]
    print(f"Fold {i}: {dates.min().date()} to {dates.max().date()} "
          f"({len(fold):,} rows)")

Fold 1: 2015-07-01 to 2015-09-18 (5,875 rows)
Fold 2: 2015-09-19 to 2015-12-06 (5,950 rows)
Fold 3: 2015-12-07 to 2016-02-23 (5,480 rows)
Fold 4: 2016-02-24 to 2016-05-12 (9,703 rows)
Fold 5: 2016-05-13 to 2016-07-31 (9,709 rows)
Fold 6: 2016-08-01 to 2016-10-18 (10,607 rows)
Fold 7: 2016-10-19 to 2017-01-05 (8,917 rows)
Fold 8: 2017-01-06 to 2017-03-25 (8,544 rows)
Fold 9: 2017-03-26 to 2017-06-12 (11,235 rows)
Fold 10: 2017-06-13 to 2017-08-31 (11,376 rows)
